# Лабораторная работа №1
## Разведочный анализ данных: Titanic

**Выполнил:** Акимов Александр, гр. 4374 

---

**Цель работы:** провести разведочный анализ данных набора данных 
Titanic, выявить закономерности, определить ключевые признаки и подготовить 
данные для построения модели машинного обучения.

**Задача:** бинарная классификация — предсказать, выжил ли пассажир 

**Источник данных:** [Kaggle — Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic)

## 1. Загрузка данных и первичный обзор

Загружаем обучающую выборку и смотрим на её структуру.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as mlt
import seaborn as sns

# читаем датасет из файла и выводим его шапку
df = pd.read_csv('data/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 1.1 Первичный обзор
Проверяем размеры датасета, типы данных признаков, целевую переменную

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


### Вывод:

**Структура данных:**
- **891 строка** — пассажиры обучающей выборки;
- **12 столбцов** — характеристики;
- **Целевая переменная:** `Survived` (0 / 1).

**Столбцы:**

| Столбец | Тип | Описание |
|---|---|---|
| `PassengerId` | int | ID пассажира |
| `Survived` | int | **Целевая**: 0 — погиб, 1 — выжил |
| `Pclass` | int | Класс билета |
| `Name` | str | Имя пассажира |
| `Sex` | str | Пол пассажира |
| `Age` | float | Возраст |
| `SibSp` | int | Количество братьев/сестёр + супругов |
| `Parch` | int | Количество родителей/детей |
| `Ticket` | str | Номер билета |
| `Fare` | float | Стоимость билета |
| `Cabin` | str | Номер каюты |
| `Embarked` | str | Порт посадки |

## 2. Разведочный анализ
**Подготавливаем данные для обучения:**
- проверка качества данных
- целевая переменная
- одномерный анализ
- двумерный анализ
- корреляции
- комбинированные признаки
- выводы

### 2.1 Проверка качества данных
Проверяем наличие:
- пропусков
- дубликатов
- аномалий

In [3]:
# Итоговая таблица пропусков
missing = {
    'Столбец': df.columns,
    'Пропусков': df.isnull().sum(),
    'Доля, %': (df.isnull().sum() / len(df)).round(2)
}

missing_df = pd.DataFrame(missing)
missing_df

,Столбец,Пропусков,"Доля, %"
PassengerId,PassengerId,0,0.00
Survived,Survived,0,0.00
Pclass,Pclass,0,0.00
Name,Name,0,0.00
Sex,Sex,0,0.00
Age,Age,177,0.20
SibSp,SibSp,0,0.00
Parch,Parch,0,0.00
Ticket,Ticket,0,0.00
Fare,Fare,0,0.00


In [4]:
# оставим только столбцы с пропусками 
missing_df = missing_df[missing_df['Пропусков'] > 0]

# отсортируем по убыванию
missing_df = missing_df.sort_values('Пропусков', ascending=False)
missing_df

,Столбец,Пропусков,"Доля, %"
Cabin,Cabin,687,0.77
Age,Age,177,0.20
Embarked,Embarked,2,0.00


#### Вывод по пропускам:
- Для `Cabin` > 75% пропусков - удаляем признак (мало информации)
- Для `Age` и `Embarked` менее 25% пропусков - заполняем медианой

In [25]:
# провека дубликатов
print(f'Дубликатов: {df.duplicated().sum()}')

Дубликатов: 0


#### Вывод по дубликатам:
- дубликатов не обнаружено

#### Поиск аномальных значений:
Проверим отдельно **числовые** и **строковые** параметры.

Будем так же смотреть на кол-во **уникальных** значений параметра в поисках ошибок

In [21]:
# разделим признаки на числовые и строковые
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
str_cols = df.select_dtypes(include=['str']).columns

# проверка аномалий числовых значений
df[num_cols].describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [43]:
# проверим уникальные значения
print(df[num_cols].nunique())

# выведем интересующие уникальные значения
for col in ['Survived', 'Pclass']:
    uniques = sorted(df[col].dropna().unique().tolist())
    print(f"{col}: уникальных = {uniques}")

PassengerId    891
Survived         2
Pclass           3
Age             88
SibSp            7
Parch            7
Fare           248
dtype: int64
Survived: уникальных = [0, 1]
Pclass: уникальных = [1, 2, 3]


#### Вывод по числовым параметрам:
- кол-во ID равно кол-ву наблюдений, в диапазоне от 1 до 891 - **без аномалий**
- кол-во уникальных значений `Survived` - 2: выжил или нет - **без аномалий**
- кол-во униклаьных значений `Pclass` - 3: 1й, 2й, 3й - **без аномалий**
- диапазон возраста от 0.42 до 80 лет - **без аномалий**
- кол-во родственников в пределах разумного значения - от 0 до 8 - **без аномалий**
- стоимость билетов разнится, но без аномальных значений - **без аномалий**

In [44]:
# проверка аномалий строковых значений
print(df[str_cols].nunique())

# выведем интересующие уникальные значения
for col in ['Sex', 'Embarked']:
    uniques = sorted(df[col].dropna().unique().tolist())
    print(f"{col}: уникальных = {uniques}")

Name        891
Sex           2
Ticket      681
Cabin       147
Embarked      3
dtype: int64
Sex: уникальных = ['female', 'male']
Embarked: уникальных = ['C', 'Q', 'S']


#### Вывод по строковым параметрам:
- кол-во имен равно кол-ву наблюдений - **без аномалий**
- кол-во уникальных значений `Sex` - 2: мужчина или женщина - **без аномалий**
- кол-во униклаьных значений `Embarked` - 3: C, Q, S - **без аномалий**

### 2.2. Целевая переменная